# Intelligent Rivers · Snow above the Beas
## A guided SYMFLUENCE experiment near Rohtang Pass

**How much snow is stored, when does it disappear, and what can a satellite actually tell us?**

We use the existing [Rohtang configuration](config_rohtang_snow.yaml) to acquire real geospatial attributes, ERA5 meteorology and MODIS observations, run a single-HRU SUMMA simulation, calibrate snow parameters against MODIS, and interpret the results. All acquisition and model execution go through **SYMFLUENCE**.

This notebook follows the structure of the [point-scale snow tutorial](../01_point_vertical_flux_estimation/01a_point_scale_snotel.ipynb) and [Provo workshop](04b_provo_river_workshop.ipynb). The YAML is the single source of experiment settings.

| Time | Activity | Question |
|---|---|---|
| 5 min | Inspect configuration and terrain | What does our one HRU represent? |
| 10 min | Follow acquisition and preprocessing | Which data are observations, and which are reanalysis? |
| 10 min | Inspect the simulated snow season | When does storage accumulate and decline? |
| 15 min | Compare with MODIS | What changes when clouds hide the ground? |
| 15 min | Calibrate and interpret | Does a better fit transfer to the next season? |

**Instructor preparation:** finish the configuration-driven workflow before the session; remote queues are not included in this 55-minute schedule. The notebook can inspect partially completed runs, but missing results are explicitly marked as unavailable. There are no synthetic data or substitute model outputs.

## 1 · Open the experiment

Use the repository's Python environment. The analysis uses NumPy, pandas, xarray, Matplotlib, GeoPandas, Rasterio and Jupyter. A fresh model run additionally needs the usual SYMFLUENCE installation, SUMMA, CDS access and NASA Earthdata credentials. Credentials stay outside the notebook.

**Two ways to use this notebook**

- Set `RUN_STEPS = False` to explore outputs prepared by the instructor. “Run All” reads the existing experiment and skips unavailable analyses with an explanation.
- The default `RUN_STEPS = True` runs the native SYMFLUENCE stages as you reach them. Start from a fresh kernel and run cells in order. Do not do this while the CLI is running the same experiment.

For an unattended run, the equivalent command from the repository root is:

```bash
symfluence workflow run --config examples/04_workshop_notebooks/config_rohtang_snow.yaml
```

In [ ]:
import os
import sys
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import geopandas as gpd
import symfluence as sf_package
from IPython.display import display, Markdown
from symfluence import SYMFLUENCE
from symfluence.core.config.models import SymfluenceConfig
from symfluence.core.mixins.project import resolve_data_subdir

RUN_STEPS = True
relative_config = Path("examples/04_workshop_notebooks/config_rohtang_snow.yaml")
anchors = [Path.cwd(), Path(sf_package.__file__).resolve().parent]
roots = [parent for anchor in anchors for parent in [anchor, *anchor.parents]]
REPO = next((root for root in roots if (root / relative_config).is_file()), None)
if REPO is None:
    raise FileNotFoundError("Open this notebook inside a SYMFLUENCE checkout with its companion YAML.")
os.chdir(REPO)  # The shared YAML deliberately uses paths relative to the repository root.
CONFIG_PATH = REPO / relative_config
config = SymfluenceConfig.from_file(CONFIG_PATH)
project = config.system.data_dir / f"domain_{config.domain.name}"
experiment = config.domain.experiment_id
forcing_dir = resolve_data_subdir(project, "forcing") / "SUMMA_input"
snow_dir = Path(config.evaluation.modis_snow.data_dir).expanduser().resolve()
sim_dir = project / "simulations" / experiment / "SUMMA"
start, end = pd.Timestamp(config.domain.time_start), pd.Timestamp(config.domain.time_end)
days = pd.date_range(start.normalize(), end.normalize(), freq="D")
spinup_end = pd.Timestamp(config.domain.spinup_period.split(",")[1].strip())
eval_start, eval_end = [pd.Timestamp(s.strip()) for s in config.domain.evaluation_period.split(",")]

BLUE, TEAL, ORANGE, GREY = "#285a7c", "#168a83", "#d97941", "#9aa6af"
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.18, "font.size": 10.5})
print(f"Python: {sys.executable}")
print(f"SYMFLUENCE: {sf_package.__version__}")
print(f"Config: {CONFIG_PATH.relative_to(REPO)}")
print(f"Config SHA-256: {hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()}")
print(f"Project: {project}")
print("Mode:", "execute native workflow stages" if RUN_STEPS else "inspect existing outputs")

In [ ]:
display(pd.DataFrame([
    ("Domain", config.domain.name),
    ("Spatial representation", f"{config.domain.definition_method}; {config.domain.discretization}"),
    ("Point (latitude / longitude)", config.domain.pour_point_coords),
    ("Bounding box (north / west / south / east)", config.domain.bounding_box_coords),
    ("Model / forcing", f"{config.model.hydrological_model} / {config.forcing.dataset}"),
    ("Simulation", f"{start} to {end}"),
    ("Spin-up", config.domain.spinup_period),
    ("Configured calibration interval", config.domain.calibration_period),
    ("Evaluation interval", config.domain.evaluation_period),
    ("MODIS products", ", ".join(config.evaluation.modis_snow.products)),
], columns=["Setting", "Value"]).set_index("Setting"))
# Inspect the actual file, rather than a second set of settings in this notebook.
display(Markdown("<details><summary>Show the shared YAML</summary>\n\n```yaml\n"
                 + CONFIG_PATH.read_text() + "\n```\n</details>"))

**Exercise 1 — Read the experiment before running it.** Find the simulation, spin-up and evaluation dates in the YAML. Why is the model started before the snow season we want to interpret? A configured calibration interval does not mean calibration was performed: this config disables automatic optimization.

The domain is a roughly 4.2 km² rectangular footprint near Rohtang Pass, represented by **one HRU** (hydrological response unit). It is not a delineated river basin or a gauged outlet. Elevation and land cover vary within it, while the model represents one effective land column.

In [ ]:
workflow = None

def run_stage(names):
    """Use SYMFLUENCE's public step interface; no independent data pipeline."""
    global workflow
    if not RUN_STEPS:
        print("Inspect mode — not executed:", ", ".join(names))
        return
    if workflow is None:
        # Figures are authored below for snow; native generic plots expect streamflow observations.
        workflow = SYMFLUENCE(config, visualize=False)
    workflow.run_individual_steps(names)

def unavailable(reason):
    display(Markdown(f"**Not available yet:** {reason}"))

def mark_periods(ax):
    ax.axvspan(start, spinup_end + pd.Timedelta(days=1), color=GREY, alpha=0.14)
    ax.axvline(eval_start, color=GREY, ls="--", lw=1)
    ax.set_xlim(start, end)
    locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

## 2 · Build and inspect the land column

These are the same native project, attribute-acquisition and domain steps used in the existing tutorials. SYMFLUENCE acquires the DEM, soil classes and MODIS land cover, then creates the point domain and its HRU.

**Predict:** Is the HRU's mean elevation likely to equal the ERA5 grid's terrain elevation? What would that difference do to snowfall and melt?

In [ ]:
run_stage(["setup_project", "create_pour_point", "acquire_attributes",
           "define_domain", "discretize_domain"])

basin_file = project / "shapefiles" / "river_basins" / f"{config.domain.name}_riverBasins_point.shp"
hru_file = (project / "shapefiles" / "catchment" / "point" / experiment
            / f"{config.domain.name}_HRUs_{config.domain.discretization}.shp")
dem_file = (resolve_data_subdir(project, "attributes") / "elevation" / "dem"
            / f"domain_{config.domain.name}_elv.tif")
basin = gpd.read_file(basin_file).to_crs(4326) if basin_file.exists() else None
hrus = gpd.read_file(hru_file) if hru_file.exists() else None
if hrus is not None:
    if len(hrus) != 1:
        raise ValueError("This lesson expects one HRU; define spatial weights before using multiple HRUs.")
    fields = [k for k in ["HRU_ID", "HRU_area", "elev_mean", "hru_type"] if k in hrus]
    display(hrus[fields])
    area_km2 = hrus.to_crs(hrus.estimate_utm_crs()).area.sum() / 1e6
    print(f"Footprint area: {area_km2:.2f} km²")
else:
    unavailable("Run the domain and discretization stages to inspect the HRU.")

In [ ]:
if basin is not None and dem_file.exists():
    import rasterio
    from rasterio.plot import show
    with rasterio.open(dem_file) as dem:
        fig, ax = plt.subplots(figsize=(9, 6), layout="constrained")
        terrain = dem.read(1, masked=True)
        show(terrain, transform=dem.transform, ax=ax, cmap="terrain")
        basin.to_crs(dem.crs).boundary.plot(ax=ax, color="#202a35", linewidth=2)
        lat, lon = map(float, config.domain.pour_point_coords.split("/"))
        point = gpd.GeoSeries(gpd.points_from_xy([lon], [lat]), crs=4326).to_crs(dem.crs)
        point.plot(ax=ax, marker="*", color=ORANGE, edgecolor="black", markersize=160)
        fig.colorbar(ax.images[0], ax=ax, label="DEM elevation (m)", shrink=0.8)
        ax.set(title="Rohtang: the model footprint", xlabel=f"X ({dem.crs})", ylabel=f"Y ({dem.crs})")
        from matplotlib.ticker import MaxNLocator, FormatStrFormatter
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.yaxis.set_major_locator(MaxNLocator(5))
        if dem.crs.is_geographic:
            ax.set(xlabel="Longitude (°E)", ylabel="Latitude (°N)")
            ax.xaxis.set_major_formatter(FormatStrFormatter("%.3f"))
            ax.yaxis.set_major_formatter(FormatStrFormatter("%.3f"))
        plt.show()
else:
    unavailable("The footprint and DEM are needed for the terrain map.")

## 3 · Acquire observations and meteorology

The config requests **MOD10A1 Collection 6.1 (Terra)** through SYMFLUENCE's AppEEARS option and **ERA5** through its CDS option. Requests can queue remotely. Run this preparation before teaching and reuse the project directory in class.

ERA5 is reanalysis, not an on-site weather station. SUMMA needs precipitation, air temperature, humidity, wind, pressure, and shortwave and longwave radiation. Model-agnostic preprocessing handles spatial matching; SUMMA-specific preprocessing writes model inputs. The configured lapse-rate adjustment is a modelling assumption worth examining.

In [ ]:
run_stage(["process_observed_data", "acquire_forcings", "model_agnostic_preprocessing"])
run_stage(["model_specific_preprocessing"])

inventory = []
for label, folder, pattern in [
    ("Raw meteorology", resolve_data_subdir(project, "forcing") / "raw_data", "*.nc"),
    ("SUMMA forcing", forcing_dir, "*.nc"),
    ("MODIS gridded observations", snow_dir, "*.nc"),
    ("SUMMA results", sim_dir, "*.nc"),
]:
    files = list(folder.glob(pattern))
    inventory.append({"Artifact": label, "Files": len(files),
                      "Size (MB)": round(sum(p.stat().st_size for p in files) / 1e6, 2),
                      "Folder": str(folder)})
display(pd.DataFrame(inventory).set_index("Artifact"))
print("A file count shows availability, not a successful or complete simulation.")

### Read the actual model inputs

The following helper reads one HRU and rejects conflicting duplicate timestamps. It does not fill data gaps or average different HRUs. Keeping the checks here makes it easier to adapt the lesson to another point experiment.

In [ ]:
def read_single_hru_series(files, variable):
    pieces, unit_set = [], set()
    for path in sorted(files):
        with xr.open_dataset(path) as ds:
            if variable not in ds or "time" not in ds[variable].dims:
                continue
            da = ds[variable]
            for dim in list(da.dims):
                if dim != "time":
                    if da.sizes[dim] != 1:
                        raise ValueError(f"{path.name}: {variable} has {da.sizes[dim]} {dim}; expected one.")
                    da = da.isel({dim: 0})
            series = da.load().to_series()
            series.index = pd.DatetimeIndex(series.index)
            pieces.append(series)
            unit_set.add(str(da.attrs.get("units", "")).strip())
    if not pieces:
        return None, None
    if len(unit_set) != 1:
        raise ValueError(f"Inconsistent {variable} units: {unit_set}")
    combined = pd.concat(pieces).sort_index()
    if combined.index.has_duplicates:
        counts = combined.groupby(level=0).nunique(dropna=False)
        if (counts > 1).any():
            raise ValueError(f"Conflicting {variable} values at duplicate times; inspect the input files.")
        combined = combined[~combined.index.duplicated()]
    return combined.loc[start:end], unit_set.pop()

# Follow SUMMA's manifest so stale inputs from other experiments are not mixed.
import shlex
forcing_list_file = project / "settings" / "SUMMA" / "forcingFileList.txt"
forcing_files = []
if forcing_list_file.exists():
    for line in forcing_list_file.read_text().splitlines():
        names = shlex.split(line.split("!")[0], comments=True)
        if names:
            forcing_files.append(forcing_dir / names[0])
    missing = [str(path) for path in forcing_files if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"SUMMA's forcing manifest references missing inputs: {missing[:3]}")
airtemp, temperature_units = read_single_hru_series(forcing_files, "airtemp")
pptrate, precip_units = read_single_hru_series(forcing_files, "pptrate")
weather = None
if airtemp is not None and pptrate is not None:
    if temperature_units not in {"K", "kelvin"} or precip_units not in {"kg m-2 s-1", "kg m^-2 s^-1"}:
        raise ValueError(f"Check forcing units before converting: {temperature_units}, {precip_units}")
    hourly = pd.concat([airtemp.rename("temperature_k"), pptrate.rename("precipitation_rate")], axis=1)
    timestep = int(config.forcing.time_step_size)
    if timestep <= 0 or 86400 % timestep:
        raise ValueError("The configured forcing timestep must divide one day exactly.")
    if len(hourly) < 2 or not np.all(np.diff(hourly.index.asi8) == timestep * 10**9):
        raise ValueError("Forcing timestamps contain gaps or do not match the configured timestep.")
    if (hourly.precipitation_rate.dropna() < -1e-10).any():
        raise ValueError("Negative precipitation rates need inspection upstream.")
    count = hourly.resample("D").count()
    weather = pd.DataFrame({
        "temperature_c": (hourly.temperature_k - 273.15).resample("D").mean(),
        "precipitation_mm": (hourly.precipitation_rate.clip(lower=0) * timestep).resample("D").sum(min_count=1),
    }).reindex(days)
    weather.loc[count.temperature_k.reindex(days).ne(86400 // timestep), "temperature_c"] = np.nan
    weather.loc[count.precipitation_rate.reindex(days).ne(86400 // timestep), "precipitation_mm"] = np.nan
    print("Daily values use the UTC dates of the model-input timestamps; incomplete days are masked.")
    display(weather.describe().round(2))
else:
    unavailable("Run acquisition and both preprocessing stages to inspect SUMMA-ready forcing.")

In [ ]:
if weather is not None:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
    axes[0].plot(weather.index, weather.temperature_c, color=ORANGE, lw=1)
    axes[0].axhline(0, color=GREY, lw=0.8)
    axes[0].set(ylabel="Daily mean temperature (°C)", title="ERA5 as supplied to SUMMA after preprocessing")
    axes[1].bar(weather.index, weather.precipitation_mm, width=1, color=BLUE)
    axes[1].set_ylabel("Precipitation (mm/day)")
    for ax in axes: mark_periods(ax)
    plt.show()
else:
    unavailable("The weather plot will appear once model-ready forcing exists.")

### Audit the grid cell and elevation correction before calibration

The earlier point extractor selected the **first stored ERA5 cell** (32.50°N, 77.00°E). The corrected native extractor chooses the nearest geographic cell to 32.37°N, 77.25°E: **32.25°N, 77.25°E**, independent of coordinate order. For the original year, the first cell was about 3.1°C colder on average. This is a spatial-selection bug, not evidence that parameters need wider bounds.

The audit below reads the acquired ERA5 grid, the native SUMMA forcing manifest, and the intersection elevations. It checks that the applied temperature difference matches `lapse_rate × (forcing elevation − HRU elevation)` and displays the actual selected coordinates. The earlier −0.28°C correction was small compared with the grid-cell error.

**Elevation limitation:** the small footprint DEM cannot establish the elevation of an ERA5 cell outside it. The native point workflow then uses equal elevations to apply **zero correction**, rather than extrapolating the local DEM. That fallback does not establish ERA5 model orography; a physically justified lapse adjustment would need that orography or other supporting data. Nearest-cell selection itself does not validate reanalysis temperature or precipitation in steep terrain.

“Precipitation during subzero hours” below is a temperature-screened forcing diagnostic, **not measured snowfall or SUMMA’s wet-bulb partitioning**. Actual SUMMA snowfall and rainfall are checked after the run.
**Cached first-season diagnostic:** with default parameters, correcting the selection reduced peak SWE from about 1,130 to 997 mm and produced a seven-day snow-layer-free window beginning 10 August 2023. The old run retained snow through August. This diagnostic isolates the preprocessing correction; the main comparison below uses the corrected baseline and the separate calibration candidate.


In [ ]:
raw_dir = resolve_data_subdir(project, "forcing") / "raw_data"
raw_path = raw_dir / f"domain_{config.domain.name}_ERA5_CDS_{start.year}_{end.year}.nc"
if not raw_path.exists():
    raise FileNotFoundError("Complete the two-season native ERA5 acquisition before the forcing audit.")
with xr.open_dataset(raw_path) as ds:
    raw = ds[["air_temperature", "precipitation_flux"]].sel(time=slice(start, end)).load()
point_lat, point_lon = map(float, config.domain.pour_point_coords.split("/"))
nearest = raw.sel(latitude=point_lat, longitude=point_lon, method="nearest")
first = raw.isel(latitude=0, longitude=0)
raw_nearest_t = nearest.air_temperature.to_series()
raw_first_t = first.air_temperature.to_series()
# Every file in the native manifest must carry corrected selection provenance.
for path in forcing_files:
    with xr.open_dataset(path) as ds:
        assert ds.attrs.get("point_extraction_method") == "nearest_coordinate_v1", path
        np.testing.assert_allclose([ds.attrs["point_forcing_latitude"], ds.attrs["point_forcing_longitude"]],
                                   [float(nearest.latitude), float(nearest.longitude)])
intersection_path = project / "shapefiles/catchment_intersection/with_forcing" / f"{config.domain.name}_ERA5_intersected_shapefile.csv"
intersection = pd.read_csv(intersection_path)
lapse = float(config.forcing.lapse_rate)
lapse_k_per_m = lapse if abs(lapse) < 0.1 else lapse / 1000
expected_correction = (intersection.weight * lapse_k_per_m *
                       (intersection.S_2_elev_m - intersection.S_1_elev_m)).sum() if config.forcing.apply_lapse_rate else 0.0
# 'airtemp' is the native hourly forcing series loaded above, before daily averaging.
temperature_check = pd.concat([airtemp.rename("input"), raw_nearest_t.rename("raw")], axis=1).dropna()
actual_correction = temperature_check.input - temperature_check.raw
np.testing.assert_allclose(actual_correction, expected_correction, atol=1e-3)
rows = []
for label, cell in [("First stored cell (old bug)", first), ("Nearest cell (current)", nearest)]:
    t = cell.air_temperature
    precip = cell.precipitation_flux
    rows.append({"cell": label, "latitude": float(cell.latitude), "longitude": float(cell.longitude),
                 "mean_temperature_C": float(t.mean())-273.15,
                 "precipitation_mm": float(precip.sum()) * config.forcing.time_step_size,
                 "precip_during_subzero_hours_mm": float(precip.where(t < 273.15, 0).sum()) * config.forcing.time_step_size})
display(pd.DataFrame(rows).set_index("cell").round(3))
display(intersection[["S_1_elev_m", "S_2_elev_m", "weight"]].rename(columns={"S_1_elev_m":"HRU_elevation_m", "S_2_elev_m":"forcing_elevation_used_m"}))
print(f"Applied correction: {actual_correction.mean():.4f} K; expected: {expected_correction:.4f} K")
print("Raw mean nearest minus first-cell temperature:", round((raw_nearest_t-raw_first_t).mean(), 3), "K")
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
x, y = np.meshgrid(raw.longitude, raw.latitude)
colours = raw.air_temperature.mean("time").transpose("latitude", "longitude") - 273.15
sc = axes[0].scatter(x, y, c=colours, s=160, cmap="coolwarm")
axes[0].scatter(point_lon, point_lat, marker="*", s=160, color="black", label="Requested point")
axes[0].scatter(float(nearest.longitude), float(nearest.latitude), marker="o", s=250,
                facecolors="none", edgecolors=TEAL, linewidths=2, label="Selected cell")
axes[0].set(xlabel="Longitude", ylabel="Latitude", title="Acquired ERA5 cell centres")
axes[0].legend(fontsize=8)
fig.colorbar(sc, ax=axes[0], label="Mean raw temperature (°C)")
axes[1].plot(raw_first_t.resample("MS").mean()-273.15, color=GREY, label="First-cell counterfactual")
axes[1].plot(raw_nearest_t.resample("MS").mean()-273.15, color=TEAL, label="Nearest cell")
axes[1].axhline(0, color=GREY, lw=0.7)
axes[1].set(ylabel="Monthly mean temperature (°C)", title="Spatial-selection effect")
mark_periods(axes[1]); axes[1].legend(fontsize=8)
plt.show()


**Exercise 2 — Read the weather.** Find a wet period below freezing and a warm period with little precipitation. Predict what SWE will do in each. Which uncertainty would matter most here: precipitation amount, temperature, or elevation adjustment? The grey band denotes spin-up; the dashed line marks the configured evaluation start.

## 4 · Run SUMMA and inspect snow storage

SUMMA solves coupled water and energy processes in the land column. We inspect its outputs directly. **SWE** is snow water equivalent: a water mass per unit area, numerically equivalent to millimetres of water for units of kg m⁻². It is not snow depth or snow-covered fraction.

\[
\Delta S = P_{snow} + F_{other,in} - M_{out} - E_{snow}
\]

This schematic budget motivates our interpretation; a daily drop in SWE alone is **not** a measurement of melt or river flow. Snowfall, sublimation and other storage exchanges can occur on the same day.

In [ ]:
run_stage(["run_model", "postprocess_results"])

# SUMMA writes variables at different frequencies to different files.
# Read daily output when available; do not accidentally choose a restart file.
all_outputs = sorted(p for p in sim_dir.glob("*.nc") if "restart" not in p.name.lower())
daily_outputs = [p for p in all_outputs if "_day" in p.stem]
timestep_outputs = [p for p in all_outputs if "_timestep" in p.stem]

def model_series(variable):
    for files in [daily_outputs, timestep_outputs]:
        series, units = read_single_hru_series(files, variable)
        if series is not None:
            return series, units
    return None, None

swe, swe_units = model_series("scalarSWE")
ground_fraction, fraction_units = model_series("scalarGroundSnowFraction")
snow = None
if swe is not None:
    if swe_units not in {"kg m-2", "kg m^-2", "mm"}:
        raise ValueError(f"Check SWE units before interpreting: {swe_units}")
    snow = pd.DataFrame({"swe_mm": swe.resample("D").last()}).reindex(days)
    if ground_fraction is not None:
        fraction = ground_fraction.resample("D").last()
        if not fraction.dropna().between(0, 1).all():
            raise ValueError("Ground snow fraction is outside [0, 1].")
        snow["ground_snow_fraction"] = fraction
    print(f"SWE available on {snow.swe_mm.notna().sum()} of {len(days)} days.")
    print("Daily points retain the UTC date of SUMMA's recorded timestamp; they are not overpass-time samples.")
    display(snow.describe().round(3))
else:
    unavailable("No scalarSWE output found. Complete the SUMMA run before interpreting snow storage.")

In [ ]:
if snow is not None and snow.swe_mm.notna().any():
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
    axes[0].fill_between(snow.index, snow.swe_mm, color=BLUE, alpha=0.2)
    axes[0].plot(snow.index, snow.swe_mm, color=BLUE)
    axes[0].set(ylabel="SWE (mm water)", title="SUMMA: simulated snow season")
    if "ground_snow_fraction" in snow:
        axes[1].plot(snow.index, snow.ground_snow_fraction, color=TEAL)
        axes[1].set(ylabel="Snow-layer presence (0/1)", ylim=(-0.03, 1.03))
    else:
        axes[1].plot(snow.index, snow.swe_mm.diff(), color=TEAL)
        axes[1].set_ylabel("Daily SWE change (mm)")
    for ax in axes: mark_periods(ax)
    plt.show()
    after_spinup = snow.loc[spinup_end + pd.Timedelta(days=1):, "swe_mm"].dropna()
    if not after_spinup.empty:
        peak_date = after_spinup.idxmax()
        print(f"Simulated peak: {after_spinup.max():.1f} mm on {peak_date.date()}")
        print("This is a model result, not an observed snow-water measurement.")
else:
    unavailable("The snow-storage plot needs non-missing SUMMA SWE.")
if snow is not None and "ground_snow_fraction" in snow:
    print("Snow diagnostic values:", np.unique(snow.ground_snow_fraction.dropna()))
source_path = config.system.data_dir / "installs/summa/build/source/engine/vegPhenlgy.f90"
if source_path.exists():
    print("Inspected SUMMA source:", source_path)
    print("Source SHA-256:", hashlib.sha256(source_path.read_bytes()).hexdigest())


### Check actual snowfall, rainfall and melt energy

The YAML asks SUMMA to record hourly `scalarSnowfall` and `scalarRainfall`, plus daily snow depth. These are the model’s precipitation partitioning results, distinct from the simple below-freezing precipitation proxy above. Integrate rates using the forcing timestep; do not interpret a daily snapshot of a flux as a daily total. A calibrated snowfall multiplier can change water input, so compare storage and snowfall as well as cover timing.

In [ ]:
snowfall_rate, snowfall_units = model_series("scalarSnowfall")
rainfall_rate, rainfall_units = model_series("scalarRainfall")
if snowfall_rate is None or rainfall_rate is None:
    raise ValueError("Rerun SUMMA preprocessing/model with SUMMA_ADDITIONAL_OUTPUTS from the YAML.")
if snowfall_units != "kg m-2 s-1" or rainfall_units != "kg m-2 s-1":
    raise ValueError("Inspect precipitation-partition units before integration.")
partition = pd.DataFrame({"snowfall_mm": snowfall_rate * config.forcing.time_step_size,
                          "rainfall_mm": rainfall_rate * config.forcing.time_step_size})
model_precipitation, _ = model_series("pptrate")
np.testing.assert_allclose(snowfall_rate + rainfall_rate, model_precipitation, atol=1e-10)
print("Baseline hourly rainfall + snowfall equals model precipitation input.")
partition_monthly = partition.resample("MS").sum(min_count=1)
display(partition_monthly.round(1))
fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
ax.bar(partition_monthly.index, partition_monthly.snowfall_mm, width=23, color=BLUE, label="SUMMA snowfall")
ax.bar(partition_monthly.index, partition_monthly.rainfall_mm, width=23,
       bottom=partition_monthly.snowfall_mm, color=TEAL, label="SUMMA rainfall")
ax.set(ylabel="Monthly water input (mm)", title="Baseline precipitation partitioning")
mark_periods(ax)
ax.legend()
plt.show()


**Exercise 3 — Explain the hydrograph of storage.** Compare the snow plot with the forcing. Identify accumulation, maximum storage, and sustained depletion. Why might positive daytime temperatures coexist with a snowpack? What independent measurement would let us check the peak SWE?

## 5 · Look at MODIS before scoring anything

MOD10A1 Collection 6.1 reports **NDSI snow cover**, an index related to snow detection. Dividing NDSI by 100 does not produce fractional snow cover. [NSIDC product documentation](https://nsidc.org/data/mod10a1/versions/61) and [NDSI versus fractional snow cover](https://nsidc.org/data/user-resources/help-center/what-ndsi-snow-cover-and-how-does-it-compare-fsc).

We therefore keep three separate quantities:

| Quantity | Meaning | Units |
|---|---|---|
| SUMMA SWE | Simulated water mass stored as snow | mm water equivalent |
| SUMMA ground snow fraction | In this build: snow-layer presence (`nSnow > 0`) | Binary 0 or 1 |
| MODIS snow-detected pixel fraction | Fraction of valid footprint pixel centres classified as snow | 0–1, conditional on clear/valid coverage |

The third quantity is an explicit observation proxy, not the mean NDSI. We classify **valid NDSI > 0** as a snow detection, with configurable thresholds below. Clouds and other invalid codes are missing, never snow-free. Pixels are selected inside the same footprint. Geographic pixels receive cosine-latitude area weights.

**Quality screening:** the completed native AppEEARS subset includes Basic QA and Algorithm Flags QA alongside NDSI. We accept Basic QA 0 (best) and 1 (good), valid NDSI codes, and sufficient footprint coverage. Algorithm flags remain available in the source for a deeper exercise; we do not impose an additional bit-mask screen here. The comparison remains exploratory because daily composites, terrain and model spatial support differ.
**Model representation audit:** the installed SUMMA source, `source/engine/vegPhenlgy.f90`, assigns `scalarGroundSnowFraction = 1` when `nSnow > 0` and zero otherwise. Trace snow without a resolved layer is excluded. This single-HRU diagnostic is **not a continuous patchy-snow depletion curve**. Comparing it with a fractional MODIS footprint primarily constrains snow-layer timing. Elevation/aspect HRUs could produce fractional basin coverage by area-weighting their separate binary states, but would require a separate spatial experiment. We do not manufacture a smooth depletion curve from SWE.


In [ ]:
MIN_VALID_FRACTION = float(config.evaluation.modis_snow.min_valid_ratio)
NDSI_THRESHOLD = float(config.evaluation.modis_snow.ndsi_threshold)  # Encoded 0–100 scale.
SNOW_FRACTION_THRESHOLD = 0.5
MAX_BASIC_QA = config.evaluation.modis_snow.max_basic_qa  # 0 = best, 1 = good.
SWE_THRESHOLD_MM = 1.0    # Used only if SUMMA ground snow fraction is unavailable.

modis_file = snow_dir / f"{config.domain.name}_MODIS_SCA_merged.nc"
modis = None
if modis_file.exists() and basin is not None:
    with xr.open_dataset(modis_file) as ds:
        if "NDSI_Snow_Cover" not in ds:
            raise ValueError("Inspect MODIS variable names; expected NDSI_Snow_Cover.")
        ndsi = ds.NDSI_Snow_Cover.load()
        if "NDSI_Snow_Cover_Basic_QA" not in ds:
            raise ValueError("Basic QA is required for this analysis; inspect the native AppEEARS output.")
        basic_qa = ds.NDSI_Snow_Cover_Basic_QA.load()
        # AppEEARS labels this axis 'julian'; retain its YYYY-MM-DD labels,
        # which match the requested product dates (not astronomical Julian days).
        product_dates = pd.to_datetime([str(t)[:10] for t in ndsi.time.values])
        ndsi = ndsi.assign_coords(time=product_dates)
        basic_qa = basic_qa.assign_coords(time=product_dates)
        lat_name = next((k for k in ["lat", "latitude"] if k in ndsi.coords), None)
        lon_name = next((k for k in ["lon", "longitude"] if k in ndsi.coords), None)
        if lat_name is None or lon_name is None:
            raise ValueError("This analysis requires the native AppEEARS geographic lat/lon output.")
        if ndsi[lat_name].ndim != 1 or ndsi[lon_name].ndim != 1:
            raise ValueError("Expected a regular geographic grid.")
        if set(ndsi.dims) != {"time", lat_name, lon_name}:
            raise ValueError(f"Unexpected MODIS dimensions: {ndsi.dims}")
        scale = ndsi.encoding.get("scale_factor", 1)
        offset = ndsi.encoding.get("add_offset", 0)
        if scale != 1 or offset != 0:
            raise ValueError("NDSI was scaled on decoding; inspect metadata before using encoded thresholds.")
        display(pd.DataFrame([{"file": modis_file.name, "variable": ndsi.name,
                               "dimensions": str(dict(ndsi.sizes)),
                               "metadata_units": ndsi.attrs.get("units", "unknown")}]))
    from shapely import intersects_xy
    lon_grid, lat_grid = np.meshgrid(ndsi[lon_name].values, ndsi[lat_name].values)
    inside = intersects_xy(basin.geometry.union_all(), lon_grid, lat_grid)
    if not inside.any():
        raise ValueError("No MODIS pixel centres inside the footprint; inspect spatial alignment.")
    weights = xr.DataArray(np.cos(np.deg2rad(lat_grid)) * inside,
        dims=[lat_name, lon_name], coords={lat_name: ndsi[lat_name], lon_name: ndsi[lon_name]})
    spatial_dims = [lat_name, lon_name]
    valid_pixel = ndsi.notnull() & (ndsi >= 0) & (ndsi <= 100) & basic_qa.isin(range(MAX_BASIC_QA + 1))
    valid_weight = weights.where(valid_pixel, 0).sum(spatial_dims)
    snow_weight = weights.where(valid_pixel & (ndsi > NDSI_THRESHOLD), 0).sum(spatial_dims)
    modis = pd.DataFrame({
        "valid_fraction": (valid_weight / weights.sum()).to_series(),
        "valid_pixels": (valid_pixel & (weights > 0)).sum(spatial_dims).to_series(),
        "snow_pixel_fraction": (snow_weight / valid_weight.where(valid_weight > 0)).to_series(),
    })
    modis.index = pd.DatetimeIndex(modis.index).normalize()
    if modis.index.has_duplicates:
        raise ValueError("Duplicate MODIS dates: inspect the merge before scoring.")
    modis = modis.reindex(days)
    modis["screened_snow_fraction"] = modis.snow_pixel_fraction.where(
        (modis.valid_fraction >= MIN_VALID_FRACTION) &
        (modis.valid_pixels >= config.evaluation.modis_snow.min_pixels))
    print(f"{inside.sum()} pixel centres in the footprint; {modis.screened_snow_fraction.notna().sum()} usable days.")
    print("Range screening and Basic QA applied; no interpolation through cloud gaps. Algorithm flags are retained in the source but not additionally screened here.")
else:
    unavailable("The native merged MODIS NetCDF and footprint are required. A raw HDF directory is not sufficient.")

In [ ]:
if modis is not None:
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, layout="constrained")
    axes[0].bar(modis.index, modis.valid_fraction, color=GREY, width=1)
    axes[0].axhline(MIN_VALID_FRACTION, color=ORANGE, ls="--", label="Minimum valid coverage")
    axes[0].set(ylabel="Valid area / footprint", ylim=(0, 1.05), title="MODIS: how much ground could we see?")
    axes[0].legend()
    axes[1].scatter(modis.index, modis.screened_snow_fraction, color=TEAL, s=16,
                    label="Snow-detected fraction of valid pixels")
    axes[1].set(ylabel="Snow-detected fraction", ylim=(-0.03, 1.03))
    axes[1].legend()
    for ax in axes: mark_periods(ax)
    plt.show()
else:
    unavailable("Satellite coverage and snow-detection plots need the processed gridded observations.")

**Exercise 4 — Make missingness visible.** Find the longest cloud/invalid-data gap during spring. Can you give an exact observed snow-disappearance date through that gap? Repeat the analysis with minimum valid coverage of 0.25, 0.5 and 0.75. Do not pick the threshold merely because it improves model agreement.

## 6 · Compare like quantities and report coverage

For a simple diagnostic, classify each sufficiently observed day as predominantly snow-detected or not, using `SNOW_FRACTION_THRESHOLD`. Compare this with SUMMA's ground snow fraction; if that output is absent, explicitly use SWE above `SWE_THRESHOLD_MM` as a different, weaker presence proxy.

We report agreement and **balanced accuracy** (the average recall for snow and no-snow days) on paired observations **within the configured evaluation period**. If either observed class is absent, balanced accuracy is undefined. Always report the paired-day count and coverage. This is a time/space/proxy comparison, not validation of SWE volume. Daily satellite composites and model snapshots are not exactly synchronized.

In [ ]:
def presence_scores(model_fraction, observations, mask):
    paired = pd.concat([model_fraction.rename("model"), observations.rename("observed")], axis=1)
    paired = paired.loc[mask].dropna()
    if paired.empty:
        return {"paired_days": 0, "agreement": np.nan, "balanced_accuracy": np.nan,
                "snow_recall": np.nan, "no_snow_recall": np.nan}
    predicted = paired.model >= SNOW_FRACTION_THRESHOLD
    observed = paired.observed >= SNOW_FRACTION_THRESHOLD
    recalls = [(predicted[observed == cls] == cls).mean() for cls in [False, True]]
    balanced = np.mean(recalls) if observed.nunique() == 2 else np.nan
    return {"paired_days": len(paired), "agreement": (predicted == observed).mean(),
            "balanced_accuracy": balanced, "snow_recall": recalls[1], "no_snow_recall": recalls[0]}

model_fraction = None
if snow is not None:
    if "ground_snow_fraction" in snow:
        model_fraction = snow.ground_snow_fraction
        model_label = "SUMMA snow-layer presence (0/1)"
    else:
        model_fraction = (snow.swe_mm > SWE_THRESHOLD_MM).astype(float).where(snow.swe_mm.notna())
        model_label = f"SUMMA SWE > {SWE_THRESHOLD_MM:g} mm (presence proxy)"
evaluation_mask = (days >= eval_start) & (days <= eval_end)
if model_fraction is not None and modis is not None:
    rows = []
    for coverage in [0.25, 0.5, 0.75]:
        observed = modis.snow_pixel_fraction.where((modis.valid_fraction >= coverage) &
            (modis.valid_pixels >= config.evaluation.modis_snow.min_pixels))
        score = presence_scores(model_fraction, observed, evaluation_mask)
        rows.append({"minimum_valid_fraction": coverage, **score,
                     "paired_fraction_of_period": score["paired_days"] / evaluation_mask.sum()})
    display(pd.DataFrame(rows).set_index("minimum_valid_fraction").round(3))
    print("Model comparator:", model_label)
    fig, ax = plt.subplots(figsize=(11, 4), layout="constrained")
    ax.plot(model_fraction.index, model_fraction, color=BLUE, label=model_label)
    ax.scatter(modis.index, modis.screened_snow_fraction, color=ORANGE, s=17,
               label="MODIS snow-detected pixel fraction")
    ax.set(ylabel="Fraction / presence proxy", ylim=(-0.04, 1.04), title="Snow coverage: model and satellite proxy")
    mark_periods(ax)
    ax.legend(loc="upper right", fontsize=9)
    plt.show()
else:
    unavailable("Both SUMMA snow output and MODIS observations are needed for comparison.")

### Put an interval around the evaluation-season melt transition

An apparent snow-to-no-snow transition is bracketed by two usable satellite dates. Missing days between them widen that bracket. We report the **last observed transition during May–August of the evaluation season**, without labelling it permanent melt-out; a later storm can restore snow. We also look for a sustained modelled snow-free window using consecutive daily values.

In [ ]:
CONSECUTIVE_DAYS = 7
melt_start = max(eval_start, pd.Timestamp(year=eval_end.year, month=5, day=1))
if model_fraction is not None:
    snow_free = (model_fraction < SNOW_FRACTION_THRESHOLD).where(model_fraction.notna()).astype(float)
    window = snow_free.rolling(CONSECUTIVE_DAYS, min_periods=CONSECUTIVE_DAYS).sum()
    # Require the complete window to lie inside the evaluation interval.
    candidate_ends = window.loc[melt_start + pd.Timedelta(days=CONSECUTIVE_DAYS - 1):eval_end]
    candidate_ends = candidate_ends[candidate_ends == CONSECUTIVE_DAYS]
    if len(candidate_ends):
        window_start = candidate_ends.index[0] - pd.Timedelta(days=CONSECUTIVE_DAYS - 1)
        print(f"First modelled {CONSECUTIVE_DAYS}-day snow-layer-free window during the May–August melt window starts {window_start.date()}.")
    else:
        print("No complete sustained modelled snow-layer-free window during the May–August melt window.")
if modis is not None:
    seen = modis.loc[melt_start:eval_end, "screened_snow_fraction"].dropna()
    previous = seen.shift(1)
    transitions = seen[(seen < SNOW_FRACTION_THRESHOLD) & (previous >= SNOW_FRACTION_THRESHOLD)]
    if len(transitions):
        right = transitions.index[-1]
        left = seen.index[seen.index.get_loc(right) - 1]
        print(f"Last observed snow-to-no-snow transition: {left.date()} to {right.date()} ({(right-left).days} days apart).")
        print("This bracket does not prove permanent melt-out or exclude a transient event inside a cloud gap.")
    else:
        print("No observed snow-to-no-snow transition can be bracketed in this interval.")
if model_fraction is None and modis is None:
    unavailable("Snow outputs are needed for the timing exercise.")

## 7 · Calibrate through SYMFLUENCE

The same YAML now includes `calibrate_model`. We use native **differential evolution (DE)** to minimize **mean absolute error (MAE)** between SUMMA’s binary snow-layer presence and the MODIS snow-pixel fraction. MAE is in fraction units (0–1); lower is better. Correlation-based scores are poorly defined when either series is constant.

We fit **1 December 2022–31 August 2023**, including the complete first summer melt, after autumn initialization. We then evaluate **1 September 2023–31 August 2024** without using those dates in parameter selection. The second season tests both onset and melt. It is a temporal holdout at the same site, with only one evaluation season; it is not evidence of transfer to other basins. Keep this split and the observation rules fixed before inspecting its scores.

The native MODIS handler applies the same footprint, area weights, NDSI threshold and Basic QA screening explained above. Missing/cloudy days stay missing. The optimizer pairs the daily outputs at their native timestamps; it does not interpolate observations across gaps. Spatial support and overpass-time differences remain.

| Parameter | Physical role | Bounds |
|---|---|---|
| `tempCritRain` | Rain/snow transition temperature | 272.16–274.16 K |
| `frozenPrecipMultip` | Snowfall input multiplier | 0.5–1.5 |
| `albedoMinSpring` | Minimum spring snow albedo | 0.30–0.55 |
| `albedoDecayRate` | Snow-albedo decay timescale | 100,000–1,000,000 s |

The bounds come from the shared YAML. They keep spring albedo below the fixed winter minimum. Binary snow-layer timing and MODIS cover alone cannot uniquely identify these parameters or SWE. A snowfall multiplier can compensate for forcing bias, so treat the fitted values as effective parameters.

**Teaching budget:** 8 initial members plus 5 generations of 8 trials, seed 42; 48 candidate simulations plus final evaluation. This small search is a demonstration, not evidence of convergence. `RUN_STEPS=True` runs it; `False` reads saved results. Baseline simulations and native final-evaluation outputs occupy separate directories. The new experiment ID also preserves the earlier one-season simulation outputs. “Calibration candidate” describes the optimizer’s returned parameter vector without implying that it is better.

**A useful mathematical limit:** for an observed snow-pixel fraction $q$ and a binary model prediction $b$, $|b-q| \geq \min(q,1-q)$. The average of that minimum on paired dates is an **observation-based binary MAE floor**. It is an oracle bound, not a forecast. Reaching it means the model picked the lower-error binary state on each paired day (either state ties at 50%); it does not recover fractional coverage, exact transition dates through clouds, or SWE. The bound does not apply to genuinely fractional predictions from multiple weighted HRUs.

The search plot shows both best-so-far MAE and the population mean (shading: ±1 population standard deviation). A flat best line can mean the initial population already found a good candidate, while other members are still improving. Inspect the floor and population spread before increasing the budget.


In [ ]:
import json
import logging
from symfluence.core.modeling.evaluators.snow import SnowEvaluator

# Check that the native calibration observations match the explained operator.
observed_path = resolve_data_subdir(project, "observations") / "snow" / "preprocessed" / f"{config.domain.name}_modis_snow_processed.csv"
if not observed_path.exists() or modis is None or model_fraction is None:
    raise FileNotFoundError("Complete observations and the baseline before calibration.")
observed_native = pd.read_csv(observed_path, parse_dates=["date"]).set_index("date").sca
pd.testing.assert_series_equal(observed_native.reindex(days),
    modis.screened_snow_fraction, check_names=False, check_freq=False,
    check_exact=False, rtol=1e-7, atol=1e-9)

snow_target = SnowEvaluator(config, project, logging.getLogger("workshop.snow"))
baseline_metrics = snow_target.calculate_metrics(sim_dir, calibration_only=False)
run_stage(["calibrate_model"])

optimization_dir = project / "optimization" / "SUMMA" / f"{config.optimization.algorithm.lower()}_{experiment}"
final_dir = optimization_dir / "final_evaluation"
best_path = optimization_dir / f"{experiment}_de_best_params.json"
final_path = optimization_dir / f"{experiment}_de_final_evaluation.json"
if not best_path.exists() or not final_path.exists():
    raise FileNotFoundError("Native calibration results are missing; inspect the workflow log.")
best = json.loads(best_path.read_text())
final = json.loads(final_path.read_text())
if not np.isfinite(best["best_score"]) or not -1 <= best["best_score"] <= 0:
    raise ValueError("Calibration did not produce a valid snow-fraction MAE; inspect the run log.")
calibrated_metrics = snow_target.calculate_metrics(final_dir, calibration_only=False)
if not calibrated_metrics or not np.isfinite(calibrated_metrics.get("Calib_MAE", np.nan)):
    raise ValueError("Final calibrated snow metrics are unavailable.")
print(f"Native result: {final_path}")
print(f"Run saved: {final['timestamp']}; seed: {config.system.random_seed}")
bounds = config.optimization.parameter_bounds
parameters = pd.DataFrame([
    {"parameter": key, "fitted": value, "lower": bounds[key][0], "upper": bounds[key][1],
     "near_bound": min(value - bounds[key][0], bounds[key][1] - value) < 0.01 * (bounds[key][1] - bounds[key][0])}
    for key, value in best["best_params"].items()
]).set_index("parameter")
display(parameters)


In [ ]:
# Use the evaluator's exact pairing for the score table, including its timestamp rounding.
calibrated_raw = snow_target.extract_simulated_data(snow_target.get_simulation_files(final_dir))
baseline_raw = snow_target.extract_simulated_data(snow_target.get_simulation_files(sim_dir))
obs_raw = observed_native.dropna()
rows = []
for label, period, prefix in [
    ("Calibration", config.domain.calibration_period, "Calib"),
    ("Second-season holdout", config.domain.evaluation_period, "Eval"),
]:
    lo, hi = [pd.Timestamp(s.strip()) for s in period.split(",")]
    obs_count = obs_raw.loc[lo:hi].dropna().size
    for run, raw, metrics in [("Baseline", baseline_raw, baseline_metrics),
                               ("Calibration candidate", calibrated_raw, calibrated_metrics)]:
        rounded = raw.copy()
        rounded.index = rounded.index.round("h")
        paired = pd.concat([rounded.rename("model"), obs_raw.rename("observed")], axis=1).loc[lo:hi].dropna()
        mae = (paired.model - paired.observed).abs().mean()
        np.testing.assert_allclose(mae, metrics[f"{prefix}_MAE"], atol=1e-9)
        observed_presence = paired.observed >= SNOW_FRACTION_THRESHOLD
        predicted_presence = paired.model >= SNOW_FRACTION_THRESHOLD
        recalls = [(predicted_presence[observed_presence == cls] == cls).mean() for cls in [False, True]]
        rows.append({"period": label, "run": run,
                     "observed_snow_days": int(observed_presence.sum()),
                     "observed_no_snow_days": int((~observed_presence).sum()),
                     "balanced_accuracy": np.mean(recalls) if observed_presence.nunique() == 2 else np.nan, "paired_days": len(paired),
                     "available_observations": obs_count, "calendar_days": (hi-lo).days + 1,
                     "binary_MAE_floor": np.minimum(paired.observed, 1-paired.observed).mean(),
                     "MAE": mae, "RMSE": metrics[f"{prefix}_RMSE"]})
comparison = pd.DataFrame(rows).set_index(["period", "run"])
display(comparison.round(4))

history_path = optimization_dir / f"{experiment}_parallel_iteration_results.csv"
history = pd.read_csv(history_path)
# Native histories may append repeated runs: show the most recent iteration-zero segment.
starts = history.index[history.iteration == 0]
if len(starts):
    history = history.loc[starts[-1]:]
calibrated_daily = calibrated_raw.resample("D").last().reindex(days)
calibrated_swe, _ = read_single_hru_series(sorted(final_dir.glob("*_day.nc")), "scalarSWE")
fig, axes = plt.subplots(3, 1, figsize=(11, 9), layout="constrained")
axes[0].plot(model_fraction.index, model_fraction, color=BLUE, label="Baseline")
axes[0].plot(calibrated_daily.index, calibrated_daily, color=TEAL, label="Calibration candidate")
axes[0].scatter(modis.index, modis.screened_snow_fraction, s=15, color=ORANGE, label="MODIS proxy")
axes[0].set(ylabel="Presence / snow-pixel fraction", ylim=(-0.04, 1.04), title="Fit and second-season holdout (dashed boundary)")
axes[0].legend(ncol=3)
mark_periods(axes[0])
axes[1].plot(snow.index, snow.swe_mm, color=BLUE, label="Baseline SWE")
if calibrated_swe is not None:
    axes[1].plot(calibrated_swe.index, calibrated_swe, color=TEAL, label="Calibration candidate SWE")
axes[1].set(ylabel="SWE (mm)", title="Changed storage is a model result; MODIS does not measure SWE")
axes[1].legend()
mark_periods(axes[1])
# SYMFLUENCE stores maximization scores: MAE is saved with a minus sign.
axes[2].plot(history.iteration, -history.score, marker="o", color=TEAL, label="Best candidate")
mean_mae = -history.mean_score
axes[2].plot(history.iteration, mean_mae, ls="--", color=GREY, label="Population mean")
axes[2].fill_between(history.iteration, (mean_mae-history.std_score).clip(lower=0),
                     mean_mae+history.std_score, color=GREY, alpha=0.15)
axes[2].axhline(baseline_metrics["Calib_MAE"], color=BLUE, ls=":", label="Baseline")
floor = comparison.loc[("Calibration", "Calibration candidate"), "binary_MAE_floor"]
axes[2].axhline(floor, color=ORANGE, ls="--", lw=1, label="Binary MAE floor")
axes[2].legend(ncol=2, fontsize=9)
axes[2].set(xlabel="DE generation (0 = initial population)", ylabel="Calibration MAE", title="Search population and binary-output limit; lower is better")
plt.show()
for prefix, label in [("Calib", "Calibration"), ("Eval", "Holdout")]:
    delta = baseline_metrics[f"{prefix}_MAE"] - calibrated_metrics[f"{prefix}_MAE"]
    print(f"{label}: MAE reduction = {delta:.4f} (positive means improvement).")

# Physical changes in the candidate are separate from its cover score.
candidate_snowfall, _ = read_single_hru_series(sorted(final_dir.glob("*_timestep.nc")), "scalarSnowfall")
candidate_rainfall, _ = read_single_hru_series(sorted(final_dir.glob("*_timestep.nc")), "scalarRainfall")
candidate_partition = pd.DataFrame({"snowfall_mm": candidate_snowfall * config.forcing.time_step_size,
                                    "rainfall_mm": candidate_rainfall * config.forcing.time_step_size})
if calibrated_swe is not None:
    storage_rows = []
    for label, period in [("Calibration", config.domain.calibration_period), ("Second-season holdout", config.domain.evaluation_period)]:
        lo, hi = [pd.Timestamp(s.strip()) for s in period.split(",")]
        for run, series in [("Baseline", swe), ("Calibration candidate", calibrated_swe)]:
            subset = series.loc[lo:hi + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)]
            inputs = partition if run == "Baseline" else candidate_partition
            totals = inputs.loc[lo:hi + pd.Timedelta(days=1) - pd.Timedelta(nanoseconds=1)].sum()
            storage_rows.append({"period": label, "run": run, "peak_SWE_mm": subset.max(),
                                 "peak_date": str(subset.idxmax().date()), "last_SWE_mm": subset.iloc[-1],
                                 "snowfall_input_mm": totals.snowfall_mm, "rainfall_input_mm": totals.rainfall_mm})
    display(pd.DataFrame(storage_rows).set_index(["period", "run"]).round(1))


**Exercise 5 — Interpret the calibration, including failure to improve.**

- Did the objective improve, and did that improvement carry into the holdout? Report paired days alongside both scores.
- Has the best score reached the binary MAE floor? Is the population spread shrinking even if the best line is flat? This SUMMA build reports binary snow-layer presence; it can remain at one while SWE changes substantially. More iterations cannot guarantee an informative objective.
- Did a parameter reach a bound? Check the forcing and model assumptions before widening it.
- Compare the two SWE curves. Can MODIS distinguish their water storage? Propose an independent observation to constrain it.
- For a larger-budget experiment, change only the iteration budget and use a distinct experiment ID in the shared YAML. Keep the observation rules and period split fixed before fitting. Do not repeatedly choose settings using the holdout score.



## 8 · Discuss the result, then change one assumption

Work in pairs and record evidence from the plots. In particular, compare July–August snow depletion in MODIS with the uncalibrated model. Check the two class recalls: a model that predicts snow on every day can have apparently good overall agreement but balanced accuracy of only 0.5 when both observed classes are present (undefined otherwise).

1. **Observation sensitivity:** rerun Sections 5–6 with NDSI thresholds of 0, 10 and 40. How many days change classification? These are alternative proxies, not automatically interchangeable product definitions.
2. **Cloud sensitivity:** explain the trade-off in the coverage table. Are the retained days representative of the whole snow season?
3. **Scale:** the HRU is one effective land column. Which effects of slope, aspect, vegetation and patchy snow are lost? How could elevation-band HRUs change the comparison?
4. **Water resources:** what would you need to add before interpreting these results as river inflow or hydropower availability? Consider rainfall, soil and groundwater storage, routing and reservoir operations.

**Do not tune to a score without checking the observation operator.** Similar snow disappearance dates can coexist with different simulated SWE. Independent SWE or snow-depth observations would add information that this MODIS comparison lacks.

### Optional AI-assisted configuration review

Give an assistant the shared YAML and the relevant SYMFLUENCE schema, then ask:

> Explain this experiment to a hydrology student. Use the forcing audit to explain whether changing LAPSE_RATE would actually change temperature here. Then propose one config-only sensitivity experiment with a supported setting. Return a minimal diff and a scientific prediction. Preserve the site and dates, use a distinct EXPERIMENT_ID for the new run, and do not invent data, configuration keys or model results.

Review the proposal together. Check the field against the schema, validate the YAML, and rerun through SYMFLUENCE. Preserve the baseline outputs and config provenance. Never put credentials in the prompt.

### A short workshop conclusion

Complete these sentences using your actual results:

- “The model accumulated snow during … and reached … mm SWE on …”
- “The satellite comparison used … of … evaluation days because …”
- “Changing the observation threshold changed …, which suggests …”
- “We can say … about snow timing, but cannot infer … about river discharge.”

### Sources and reproducibility

- [Shared experiment configuration](config_rohtang_snow.yaml); its SHA-256 and the SYMFLUENCE version are printed above. Framework run logs and copied configurations are under the project's `_workLog_rohtang_snow` directory.
- [ERA5 hourly single-level data](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels), Copernicus Climate Change Service, [doi:10.24381/cds.adbb2d47](https://doi.org/10.24381/cds.adbb2d47). Meteorology is reanalysis and may be modified by the configured preprocessing.
- Hall, D. K. & Riggs, G. A. (2021), MODIS/Terra Snow Cover Daily L3 Global 500m SIN Grid, Version 61, [doi:10.5067/MODIS/MOD10A1.061](https://doi.org/10.5067/MODIS/MOD10A1.061). The config specifies the geographic and temporal subset; record the completed acquisition date when distributing results.
- [SUMMA documentation](https://summa.readthedocs.io/): model structure and output-variable interpretation.

**Data preparation:** the native SYMFLUENCE run acquired the real Rohtang inputs and completed SUMMA for September 2022–August 2024. Use the saved cell outputs and run logs to distinguish model results from satellite observations; calibration scores describe this demonstration, not an independently validated forecast.